# Neural Codec Experiment Analysis

Analysis of autonomous codec optimization results from `results.tsv`.

In [ ]:
import csv
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

# Load the TSV (tab-separated, 7 columns: datetime, score, psnr, rate, memory_gb, status, description)
rows = []
with open("results.tsv") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        row["score"] = float(row["score"])
        row["psnr"] = float(row["psnr"])
        row["rate"] = float(row["rate"])
        row["memory_gb"] = float(row["memory_gb"])
        row["status"] = row["status"].strip().upper()
        row["dt"] = datetime.strptime(row["datetime"], "%Y-%m-%d %H:%M:%S")
        rows.append(row)

print(f"Total experiments: {len(rows)}")
for i, row in enumerate(rows[:10]):
    print(f"  {i:3d}  {row['status']:8s}  score={row['score']:.2f}  psnr={row['psnr']:.2f}  rate={row['rate']:.4f}  mem={row['memory_gb']:.1f}GB  {row['description']}")

In [ ]:
from collections import Counter

counts = Counter(r["status"] for r in rows)
print("Experiment outcomes:")
for status, count in counts.most_common():
    print(f"  {status}: {count}")

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = [r for r in rows if r["status"] == "KEEP"]
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in enumerate(kept):
    print(f"  #{i:3d}  score={row['score']:.2f}  mem={row['memory_gb']:.1f}GB  {row['description']}")

## Score Over Time

Track how the best (kept) score evolves as experiments progress. The running maximum shows the "frontier" — the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = [(i, r) for i, r in enumerate(rows) if r["status"] != "CRASH"]

baseline_score = valid[0][1]["score"]

# Plot discarded as faint background dots
disc_x = [i for i, r in valid if r["status"] == "DISCARD"]
disc_y = [r["score"] for i, r in valid if r["status"] == "DISCARD"]
ax.scatter(disc_x, disc_y, c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_x = [i for i, r in valid if r["status"] == "KEEP"]
kept_y = [r["score"] for i, r in valid if r["status"] == "KEEP"]
ax.scatter(kept_x, kept_y, c="#2ecc71", s=50, zorder=4, label="Kept",
           edgecolors="black", linewidths=0.5)

# Running maximum step line
if kept_y:
    running_max = np.maximum.accumulate(kept_y)
    ax.step(kept_x, running_max, where="post", color="#27ae60",
            linewidth=2, alpha=0.7, zorder=3, label="Running best")
    best = max(kept_y)
else:
    best = baseline_score

# Label each kept experiment with its description
for x, y, r in [(i, r["score"], r) for i, r in valid if r["status"] == "KEEP"]:
    desc = r["description"].strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (x, y),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(rows)
n_kept = len(kept_y)
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Score (higher is better)", fontsize=12)
ax.set_title(f"Codec Challenge Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

margin = (best - baseline_score) * 0.15 if best > baseline_score else 1.0
ax.set_ylim(baseline_score - margin, best + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Score Over Time (with Peak Tracking)

All runs plotted by wall-clock time, with a red dashed line tracking the historical peak score.

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(16, 8))

# All non-crash runs sorted by time
valid = [r for r in rows if r["status"] != "CRASH"]
times = [r["dt"] for r in valid]
scores = [r["score"] for r in valid]

# Running peak
peak = []
best = -float("inf")
for s in scores:
    best = max(best, s)
    peak.append(best)

# Scatter all runs
ax.scatter(times, scores, color="steelblue", s=40, zorder=3, label="Run score")

# Red dashed peak line
ax.step(times, peak, where="post", color="red", linestyle="--", linewidth=1.5,
        zorder=2, label="Historical peak")

# Annotate with descriptions
for r in valid:
    desc = r["description"].strip()
    if desc:
        if len(desc) > 40:
            desc = desc[:37] + "..."
        ax.annotate(desc, (r["dt"], r["score"]),
                    textcoords="offset points", xytext=(6, 6),
                    fontsize=7, color="gray", rotation=20)

ax.set_xlabel("Time (UTC)", fontsize=12)
ax.set_ylabel("Score (higher is better)", fontsize=12)
ax.set_title("Score Over Time with Peak Tracking", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
fig.autofmt_xdate(rotation=30)

plt.tight_layout()
plt.savefig("scores.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to scores.png")

## Summary Statistics

In [ ]:
kept = [r for r in rows if r["status"] == "KEEP"]
baseline_score = rows[0]["score"]
best_score = max(r["score"] for r in kept)
best_row = [r for r in kept if r["score"] == best_score][0]

print(f"Baseline score:    {baseline_score:.2f}")
print(f"Best score:        {best_score:.2f}")
print(f"Total improvement: +{best_score - baseline_score:.2f}")
print(f"Best experiment:   {best_row['description']}")
print(f"Best PSNR:         {best_row['psnr']:.2f} dB")
print(f"Best rate:         {best_row['rate']:.4f} bpppc")
print()

# How many experiments to find each improvement
print("Cumulative improvements:")
for i, row in enumerate(kept):
    orig_idx = rows.index(row)
    print(f"  Experiment #{orig_idx:3d}: score={row['score']:.2f}  psnr={row['psnr']:.2f}  rate={row['rate']:.4f}  {row['description']}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's score
kept = [r for r in rows if r["status"] == "KEEP"]
deltas = []
for i in range(1, len(kept)):
    delta = kept[i]["score"] - kept[i-1]["score"]
    deltas.append((delta, kept[i]))

# Sort by delta improvement (biggest first)
deltas.sort(key=lambda x: x[0], reverse=True)

print(f"{'Rank':>4}  {'Delta':>8}  {'Score':>8}  Description")
print("-" * 80)
for rank, (delta, row) in enumerate(deltas, 1):
    print(f"{rank:4d}  {delta:+.2f}    {row['score']:.2f}    {row['description']}")

total_delta = sum(d for d, _ in deltas)
print(f"\n{'':>4}  {total_delta:+.2f}    {'':>8}  TOTAL improvement over baseline")